# AudioMC run inspector

Browse bundled Qwen3-Omni eval runs with full conversation context:

- **User turns**: audio player (only when wav exists locally) + reference transcript
- **Assistant turns**: seeded reference text (history)
- **Model response**: generated text on the final turn
- **LLM summary**: overall context / question / answer / GT
- **Judge**: rubric pass/fail + explanations

## Setup

See [README.md](README.md) for full instructions. Quick start:

```bash
git clone https://github.com/Satyam52/audiomc.git && cd audiomc
conda create -n audiomc-inspect python=3.11 -y
conda activate audiomc-inspect
pip install -r requirements.txt
jupyter notebook inspect_runs.ipynb
```

**Optional:** see [README.md](README.md#3-optional-download-audio-from-hugging-face) to export audio from [ScaleAI/audiomc](https://huggingface.co/datasets/ScaleAI/audiomc) into `audiomc/data/audio/`.

Run all cells, then use the **Model** dropdown to switch between `Qwen3-Omni-Instruct` and `Qwen3-Omni-Thinker`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from IPython.display import Audio, HTML, Markdown, display

ROOT = Path.cwd()
METADATA_PATH = ROOT / "metadata.jsonl"
RUN_DIRS = ["Qwen3-Omni-Instruct", "Qwen3-Omni-Thinker"]
AUDIO_ROOT = ROOT / "audiomc" / "data" / "audio"

print(f"ROOT={ROOT}")
print(f"metadata exists: {METADATA_PATH.exists()}")
print(f"audio root exists: {AUDIO_ROOT.exists()}")

ROOT=/Users/abhishek/Desktop/Code/audiomc-clone-test
metadata exists: True
audio root exists: False


In [2]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def discover_runs(root: Path = ROOT) -> List[Dict[str, Any]]:
    runs: List[Dict[str, Any]] = []
    for name in RUN_DIRS:
        d = root / name
        judged = d / "judged.jsonl"
        preds = d / "predictions.jsonl"
        if judged.exists() and preds.exists():
            runs.append(
                {
                    "run_id": name,
                    "dir": d,
                    "judged": judged,
                    "predictions": preds,
                    "summary": d / "all.json" if (d / "all.json").exists() else None,
                }
            )
    return runs


def extract_summary(rec: Dict[str, Any]) -> Optional[Dict[str, str]]:
    direct = rec.get("summary")
    if isinstance(direct, dict):
        return {
            "context": str(direct.get("context") or ""),
            "question": str(direct.get("question") or ""),
            "answer": str(direct.get("answer") or ""),
            "gt": str(direct.get("gt") or ""),
        }

    turn_summaries = rec.get("turn_summaries") or []
    if not turn_summaries:
        return None
    last = turn_summaries[-1]
    return {
        "context": str(last.get("context") or ""),
        "question": str(last.get("question") or ""),
        "answer": str(last.get("answer") or ""),
        "gt": str(last.get("gt") or ""),
    }


def load_summaries_for_run(run: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    path = run["dir"] / "summaries.jsonl"
    if not path.exists():
        return {}
    return {row["id"]: row for row in load_jsonl(path)}


def resolve_audio_path(path: str) -> str:
    if not path:
        return ""
    p = Path(path)
    if p.exists():
        return str(p)

    marker = "/data/audio/"
    normalized = str(path).replace("\\", "/")
    if marker in normalized:
        rel = normalized.split(marker, 1)[1]
        local = AUDIO_ROOT / rel
        if local.exists():
            return str(local)
    return ""


def build_turns(meta: Dict[str, Any]) -> List[Dict[str, Any]]:
    last = int(meta.get("last_user_turn") or 0)
    audio_paths = {int(k): Path(v) for k, v in (meta.get("audio_paths") or {}).items()}
    turns: List[Dict[str, Any]] = []

    for t in range(1, last + 1):
        turns.append(
            {
                "turn": t,
                "role": "user",
                "audio_path": resolve_audio_path(str(audio_paths.get(t) or "")),
                "transcript": str(meta.get(f"user_turn_{t}_transcript") or "").strip(),
            }
        )
        if t < last:
            turns.append(
                {
                    "turn": t,
                    "role": "assistant",
                    "audio_path": "",
                    "transcript": str(meta.get(f"assistant_turn_{t}_transcript") or "").strip(),
                }
            )
    return turns


def merge_example(meta: Dict[str, Any], pred: Dict[str, Any], judged: Dict[str, Any]) -> Dict[str, Any]:
    rubric_results = judged.get("rubric_results") or []
    failed = [r for r in rubric_results if not r.get("criteria_met")]
    return {
        "id": meta["id"],
        "axis": meta.get("axis"),
        "last_user_turn": meta.get("last_user_turn"),
        "rubrics": meta.get("rubrics") or [],
        "turns": build_turns(meta),
        "model_target": pred.get("model_target"),
        "model_response": judged.get("model_response") or pred.get("model_response"),
        "latency_s": pred.get("latency_s"),
        "pred_error": pred.get("error") or judged.get("pred_error"),
        "pass": judged.get("pass"),
        "judge": judged.get("judge"),
        "rubric_results": rubric_results,
        "failed_rubrics": failed,
        "n_failed_rubrics": len(failed),
        "n_rubrics": len(rubric_results),
    }


def attach_summaries(merged: List[Dict[str, Any]], run: Dict[str, Any]) -> None:
    by_id = load_summaries_for_run(run)
    for ex in merged:
        rec = by_id.get(ex["id"])
        if not rec:
            ex["llm_summary"] = None
            continue
        ex["llm_summary"] = {
            "summary": extract_summary(rec),
            "error": rec.get("error"),
            "summarizer": rec.get("summarizer"),
        }


def load_merged_run(run: Dict[str, Any], metadata_path: Path = METADATA_PATH) -> List[Dict[str, Any]]:
    meta_by_id = {row["id"]: row for row in load_jsonl(metadata_path)}
    judged_by_id = {row["id"]: row for row in load_jsonl(run["judged"])}
    pred_rows = {row["id"]: row for row in load_jsonl(run["predictions"])}

    merged: List[Dict[str, Any]] = []
    for ex_id, judged in judged_by_id.items():
        meta = meta_by_id.get(ex_id)
        if meta is None:
            continue
        pred = pred_rows.get(ex_id, judged)
        merged.append(merge_example(meta, pred, judged))
    attach_summaries(merged, run)
    return merged


def export_merged_jsonl(run: Dict[str, Any], out_path: Optional[Path] = None) -> Path:
    merged = load_merged_run(run)
    if out_path is None:
        out_path = run["dir"] / "merged.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for row in merged:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(merged)} examples -> {out_path}")
    return out_path


runs = discover_runs()
print(f"Found {len(runs)} run(s)")
for r in runs:
    print(f"  - {r['run_id']}")

Found 2 run(s)
  - Qwen3-Omni-Instruct
  - Qwen3-Omni-Thinker


In [3]:
def get_overall_summary(ex: Dict[str, Any]) -> Optional[Dict[str, str]]:
    llm = ex.get("llm_summary") or {}
    return llm.get("summary")


def render_summary_block(summary: Optional[Dict[str, str]]) -> None:
    display(Markdown("## Summary"))
    if not summary:
        display(Markdown("_Summary not available for this example._"))
        return
    display(
        Markdown(
            "\n".join(
                [
                    f"1. **Context:** {summary.get('context') or '(empty)'}",
                    f"2. **Question:** {summary.get('question') or '(empty)'}",
                    f"3. **Answer:** {summary.get('answer') or '(empty)'}",
                    f"4. **GT:** {summary.get('gt') or '(empty)'}",
                ]
            )
        )
    )


def render_turn(turn: Dict[str, Any]) -> None:
    role = turn["role"]
    t = turn["turn"]
    transcript = turn.get("transcript") or "(empty)"

    if role == "user":
        display(Markdown(f"### User turn {t}"))
        audio_path = turn.get("audio_path") or ""
        if audio_path:
            display(Audio(filename=audio_path))
        display(Markdown(f"**Transcript (reference):** {transcript}"))
    else:
        display(Markdown(f"### Assistant turn {t} (seeded history)"))
        display(Markdown(transcript))


def render_example(ex: Dict[str, Any]) -> None:
    status = "PASS" if ex.get("pass") else "FAIL"
    color = "#1b7f3a" if ex.get("pass") else "#b42318"
    display(
        HTML(
            f"""
            <div style='border:1px solid #ddd;border-radius:8px;padding:12px;margin:8px 0;'>
              <div><b>ID</b>: <code>{ex['id']}</code></div>
              <div><b>Axis</b>: {ex.get('axis')}</div>
              <div><b>Model</b>: {ex.get('model_target') or '(unknown)'}</div>
              <div><b>Judge</b>: <span style='color:{color};font-weight:700'>{status}</span>
                ({ex.get('n_rubrics', 0) - ex.get('n_failed_rubrics', 0)}/{ex.get('n_rubrics', 0)} rubrics)</div>
              <div><b>Latency</b>: {ex.get('latency_s')}</div>
            </div>
            """
        )
    )

    render_summary_block(get_overall_summary(ex))

    if ex.get("llm_summary") and ex["llm_summary"].get("error"):
        display(Markdown(f"_Summarizer error: `{ex['llm_summary']['error']}`_"))

    display(Markdown("## Conversation"))
    for turn in ex.get("turns", []):
        render_turn(turn)

    display(Markdown("## Model response (generated)"))
    display(Markdown(ex.get("model_response") or "_empty_"))

    if ex.get("pred_error"):
        display(Markdown(f"**Inference error:** `{ex['pred_error']}`"))

    display(Markdown("## Rubrics"))
    for i, rubric in enumerate(ex.get("rubrics") or [], start=1):
        display(Markdown(f"{i}. {rubric}"))

    display(Markdown("## Judge results"))
    for r in ex.get("rubric_results") or []:
        ok = r.get("criteria_met")
        mark = "✅" if ok else "❌"
        display(Markdown(f"### {mark} {r.get('rubric_item')}"))
        display(Markdown(f"_{r.get('explanation')}_"))

In [4]:
import ipywidgets as widgets

if not runs:
    raise RuntimeError("No runs found. Expected Qwen3-Omni-Instruct/ and Qwen3-Omni-Thinker/ in repo root.")

run_ids = [r["run_id"] for r in runs]
run_by_id = {r["run_id"]: r for r in runs}

run_dd = widgets.Dropdown(options=run_ids, description="Model:")
axis_dd = widgets.Dropdown(options=["(all)"], description="Axis:")
pass_dd = widgets.Dropdown(options=["(all)", "pass", "fail"], description="Result:")
search_txt = widgets.Text(description="Search:", placeholder="id / transcript / rubric")
idx_slider = widgets.IntSlider(value=0, min=0, max=0, description="Example")
prev_btn = widgets.Button(description="Prev")
next_btn = widgets.Button(description="Next")
export_btn = widgets.Button(description="Export merged.jsonl")
out = widgets.Output()

merged_cache: Dict[str, List[Dict[str, Any]]] = {}


def current_merged() -> List[Dict[str, Any]]:
    run_id = run_dd.value
    if run_id not in merged_cache:
        merged_cache[run_id] = load_merged_run(run_by_id[run_id])
    return merged_cache[run_id]


def filtered_examples() -> List[Dict[str, Any]]:
    rows = current_merged()
    axis = axis_dd.value
    result = pass_dd.value
    q = search_txt.value.strip().lower()

    out_rows = []
    for ex in rows:
        if axis != "(all)" and ex.get("axis") != axis:
            continue
        if result == "pass" and not ex.get("pass"):
            continue
        if result == "fail" and ex.get("pass"):
            continue
        if q:
            blob = json.dumps(ex, ensure_ascii=False).lower()
            if q not in blob:
                continue
        out_rows.append(ex)
    return out_rows


def refresh_axis_options(*_):
    axes = sorted({ex.get("axis") for ex in current_merged() if ex.get("axis")})
    axis_dd.options = ["(all)"] + axes


def refresh_view(*_):
    rows = filtered_examples()
    if not rows:
        with out:
            out.clear_output(wait=True)
            display(Markdown("_No examples match filters._"))
        idx_slider.max = 0
        idx_slider.value = 0
        return

    idx_slider.max = max(0, len(rows) - 1)
    if idx_slider.value > idx_slider.max:
        idx_slider.value = 0

    ex = rows[idx_slider.value]
    with out:
        out.clear_output(wait=True)
        display(Markdown(f"**Showing {idx_slider.value + 1}/{len(rows)}**"))
        render_example(ex)


def on_run_change(*_):
    refresh_axis_options()
    refresh_view()


def on_prev(_):
    idx_slider.value = max(0, idx_slider.value - 1)


def on_next(_):
    idx_slider.value = min(idx_slider.max, idx_slider.value + 1)


def on_export(_):
    with out:
        export_merged_jsonl(run_by_id[run_dd.value])


run_dd.observe(on_run_change, names="value")
axis_dd.observe(refresh_view, names="value")
pass_dd.observe(refresh_view, names="value")
search_txt.observe(refresh_view, names="value")
idx_slider.observe(refresh_view, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
export_btn.on_click(on_export)

controls = widgets.VBox(
    [
        widgets.HBox([run_dd, export_btn]),
        widgets.HBox([axis_dd, pass_dd, search_txt]),
        widgets.HBox([prev_btn, idx_slider, next_btn]),
        out,
    ]
)

display(controls)
on_run_change()

## Quick table view

Sortable overview of the selected model run (failed rubric count, axis, pass/fail).

In [5]:
rows = current_merged()
df = pd.DataFrame(
    [
        {
            "id": ex["id"],
            "axis": ex.get("axis"),
            "pass": ex.get("pass"),
            "failed_rubrics": ex.get("n_failed_rubrics"),
            "n_rubrics": ex.get("n_rubrics"),
            "latency_s": ex.get("latency_s"),
            "last_user_turn": ex.get("last_user_turn"),
            "model_response_preview": (ex.get("model_response") or "")[:120],
        }
        for ex in rows
    ]
)
df.sort_values(["pass", "failed_rubrics"], ascending=[True, False]).head(30)

,id,axis,pass,failed_rubrics,n_rubrics,latency_s,last_user_turn,model_response_preview
321,h91zc9n14t4p8398,INSTRUCTION_RETENTION,False,12,12,1.228736,3,"Yes, you're absolutely right! Point Reyes Nati..."
206,xxk7ekr27hky8oum,SELF_COHERENCE,False,10,10,8.413541,6,Absolutely. Here is a speech that weaves toget...
213,h34kmic3fslavjy7,VOICE_EDITING,False,9,11,0.949379,8,"Yes, broccoli was already added. Let me update..."
264,pjs2ybjw3d9n59fd,VOICE_EDITING,False,8,10,2.076072,8,
74,ptapql63airgq2mp,SELF_COHERENCE,False,7,10,4.168785,4,"Oh, I love that you're so excited about the sn..."
363,24tvt9iq359qqd6j,VOICE_EDITING,False,7,7,3.784766,8,"Absolutely! Here's a clear, organized list of ..."
145,bt9bs80mqz7m1qqw,INSTRUCTION_RETENTION,False,6,9,2.267740,3,You're absolutely right — Tobey Maguire was in...
263,kfse9g8j7e5mzj4w,VOICE_EDITING,False,6,10,1.734513,4,"Alright, I've processed all those changes. Her..."
291,el694m5cgk0rdi6d,INSTRUCTION_RETENTION,False,6,9,1.137398,3,You're absolutely right — *National Lampoon's ...
105,aoauzvcx5bm3qijz,SELF_COHERENCE,False,5,10,1.172076,7,Absolutely! Here's a complete list of all the ...
